# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available at the URL below.

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset source URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, including their `@id` fields and a preview of their fields and columns.

*Note: All record sets, fields, and columns are referenced strictly by their Croissant `@id` as required for reproducibility.*

In [ ]:
# Display all record sets with their IDs and their available fields/columns
print("Record Sets in this dataset:")
record_sets = list(dataset.record_sets)

for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', rs['@id'])}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields/Columns:")
    for field in fields:
        if isinstance(field, str):
            print(f"    - {field}")
        elif isinstance(field, dict):
            print(f"    - {field.get('@id', field)}")
        else:
            print(f"    - Unknown structure: {field}")

## 3. Data Extraction
Load data from selected record sets into dataframes for analysis.

**Note:** All access is by `@id` fields as displayed above. Update the below variable assignments if the recordset IDs differ.

In [ ]:
# Build a list of record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
# For demonstration, use the first record set if available
print("Record set @id's found:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"Records loaded: {len(df)}\n")
    dataframes[record_set_id] = df

# Pick one record set to explore further (the first one):
if len(record_set_ids) == 0:
    raise RuntimeError("No record set found in dataset metadata.")
main_record_set_id = record_set_ids[0]
print("Columns in the selected DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA such as filtering, normalization, and group summary using chosen numeric and group fields.

_All field/column access is via their Croissant `@id`, as per best practice._

In [ ]:
# Choose a numeric field (column) for EDA
# Replace with an available numeric column's @id shown in section 3
df = dataframes[main_record_set_id]
print("Available fields:", df.columns.tolist())

# Example: select the first numeric field (modify as needed)
numeric_field_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
if not numeric_field_candidates:
    # Try to infer numeric columns
    numeric_field_candidates = []
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna()).astype(float)
            numeric_field_candidates.append(col)
        except Exception:
            continue
if not numeric_field_candidates:
    raise ValueError("No numeric fields found. Adjust field selection for further EDA.")
numeric_field_id = numeric_field_candidates[0]
print(f"Using numeric field for EDA: {numeric_field_id}")

# Filter rows where field > threshold
threshold = df[numeric_field_id].dropna().astype(float).quantile(0.1) if not df[numeric_field_id].dropna().empty else 0.0
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field if available (e.g., a categorical field)
group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field = group_candidates[0] if group_candidates else None
if group_field:
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).agg({numeric_field_id: ['mean', 'std'], f"{numeric_field_id}_normalized": "mean"})
    print("Grouped summary:")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of a numeric field and/or its relationship to a group field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(8, 4))
pd.to_numeric(df[numeric_field_id], errors='coerce').hist(bins=40, alpha=0.7)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If group_field exists and is not too high-cardinality, make a boxplot
if group_field and df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We have loaded and accessed data from a Croissant-structured dataset using `mlcroissant`.
- All record sets and fields are referenced by their `@id` to ensure reproducibility.
- The provided EDA and visualizations offer an entry point for further biological or statistical analysis.

👉 To go further, review the record set/field IDs from the overview (Section 2) and customize the analysis or visualizations for your research questions!